# CADIP staging and AUXIP staging on-demand flows

Demonstration of flows defined on those two stories:   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-715   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-718   

## 1 - Initialisation

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Choose prefect deployment method
from resources.widget_utils import *
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
#display(dask_cluster_staging)  ==> Because it runs in a separate process we can't display staging clusters anymore

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT26_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Check that it is empty
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
assert not list(items)

# Other test values
SESSION_ID = "S1A_20200105072204051312"
CADIP_COLLECTION_ID = "sgs_sentinel1"

In [ ]:
# Other imports
import os
from pystac import ItemCollection
from rs_common.prefect_utils import *

## 2 - Set up flow parameters

In [ ]:
cadip_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

auxip_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "start_datetime": "2024-05-27T09:44:09.509000Z",
  "end_datetime": "2024-05-27T09:44:19.509000Z",
  "product_type": "AUX_PP2",
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

## 3 - Deploy Prefect flows

In [ ]:
# Deploy the Prefect flow
cadip_deploy, auxip_deploy = await deploy_prefect(
    deploy_file="./cadip_auxip_staging_flows.yaml", 
    s3_code_folder=f"users/{OWNER_ID}/code", 
    work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"]
)

## 4 - Run flows

Run one flow for CADIP staging and one for AUXIP staging.

In [ ]:
from rs_workflows.on_demand_processing import on_demand_cadip_staging
await run_prefect(
    deploy_name=cadip_deploy, 
    py_func=on_demand_cadip_staging, 
    params=cadip_flow_parameters
)

In [ ]:
from rs_workflows.auxip_flow import on_demand_auxip_staging
await run_prefect(
    deploy_name=auxip_deploy, 
    py_func=on_demand_auxip_staging, 
    params=auxip_flow_parameters
)

In [ ]:
# Processed items published to the catalog
ItemCollection(list(catalog_client.get_items(CATALOG_COLLECTION_ID)))

## 5 - Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    # Shutdown dask cluster staging
    shutdown_dask_cluster_staging()
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.